# NYT + Open Library End-to-End Smoke Test

This notebook tests the full pipeline for one NYT published date:
1. Pull NYT bestsellers for a single date.
2. Upsert into `nyt_entries`.
3. Enrich each ISBN13 via Open Library.
4. Validate joined output from `nyt_entries` + `openlibrary_enrichment`.


In [1]:
from __future__ import annotations

from pathlib import Path
import sys

import pandas as pd

project_root = Path('/Users/zacurbiztondo/literary-analysis')
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.config import settings
from src.ingest.http import HttpClient
from src.ingest.nyt import NytClient, NytConfig
from src.ingest.openlibrary import OpenLibraryClient, OpenLibraryConfig
from src.ingest.repo import Repo
from src.utils.io import connect_sqlite

settings.ensure_dirs()

http = HttpClient(
    cache_path=settings.http_cache_path,
    expire_seconds=settings.http_cache_expire_seconds,
    contact_email=settings.contact_email,
)

nyt = NytClient(http=http, cfg=NytConfig(api_key=settings.nyt_api_key, rps=settings.nyt_rps))
openlibrary = OpenLibraryClient(http=http, cfg=OpenLibraryConfig(rps=settings.openlibrary_rps))

conn = connect_sqlite(settings.db_path)
repo = Repo(conn)
repo.init_schema()

print(f"DB path: {settings.db_path}")


DB path: /Users/zacurbiztondo/literary-analysis/data/interim/books.db


In [4]:
pd.read_sql_query(
    """
    SELECT DISTINCT list_name
    FROM nyt_entries
    """,
    conn
)


,list_name
0,"Advice, How-To & Miscellaneous"
1,Audio Fiction
2,Audio Nonfiction
3,Business
4,Children’s & Young Adult Series
5,Children’s Middle Grade Hardcover
6,Children’s Picture Books
7,Combined Print & E-Book Fiction
8,Combined Print & E-Book Nonfiction
9,Graphic Books and Manga


In [2]:
# Pick one NYT published date for the smoke test.
# Change this value as needed.
published_date = '2024-06-16'

entries = nyt.fetch_lists_for_date(published_date)
upserts = repo.upsert_nyt_entries(entries)

print(f"Fetched NYT entries: {len(entries)}")
print(f"Upserted NYT entries: {upserts}")

pd.read_sql_query(
    """
    SELECT list_name, published_date, rank, title, author, isbn13
    FROM nyt_entries
    WHERE published_date = ?
    ORDER BY list_name, rank
    LIMIT 30
    """,
    conn,
    params=(published_date,),
)


Fetched NYT entries: 230
Upserted NYT entries: 230


,list_name,published_date,rank,title,author,isbn13
0,"Advice, How-To & Miscellaneous",2024-06-16,1,ATOMIC HABITS,James Clear,9780735211292
1,"Advice, How-To & Miscellaneous",2024-06-16,2,GOOD ENERGY,Casey Means with Calley Means,9780593712641
2,"Advice, How-To & Miscellaneous",2024-06-16,3,YOU DESERVE GOOD GELATO,Kacie Rose,9780593840436
3,"Advice, How-To & Miscellaneous",2024-06-16,4,THE NEW MENOPAUSE,Mary Claire Haver,9780593796252
4,"Advice, How-To & Miscellaneous",2024-06-16,5,MAKE YOUR BED,William H. McRaven,9781455570249
5,"Advice, How-To & Miscellaneous",2024-06-16,6,THE OFFICIAL STARDEW VALLEY COOKBOOK,ConcernedApe and Ryan Novak,9781984862051
6,"Advice, How-To & Miscellaneous",2024-06-16,7,THE CREATIVE ACT,Rick Rubin with Neil Strauss,9780593652886
7,"Advice, How-To & Miscellaneous",2024-06-16,8,"THE BOY, THE MOLE, THE FOX AND THE HORSE",Charlie Mackesy,9780062976581
8,"Advice, How-To & Miscellaneous",2024-06-16,9,"CHEAPER, FASTER, BETTER",Tom Steyer,9781954118645
9,"Advice, How-To & Miscellaneous",2024-06-16,10,MILLIONAIRE MISSION,Brian Preston,9781637745014


In [3]:
# Enrich only ISBN13s from the chosen NYT date.
isbn_df = pd.read_sql_query(
    """
    SELECT DISTINCT isbn13
    FROM nyt_entries
    WHERE published_date = ?
      AND isbn13 IS NOT NULL
      AND TRIM(isbn13) <> ''
    ORDER BY isbn13
    """,
    conn,
    params=(published_date,),
)

isbn_values = isbn_df['isbn13'].tolist()
print(f"Distinct ISBN13s to enrich: {len(isbn_values)}")

rows = []
failures = 0
for isbn13 in isbn_values:
    row = openlibrary.fetch_isbn13_work(isbn13)
    rows.append(row)
    if row.last_error:
        failures += 1

repo.upsert_openlibrary_enrichment(rows)
print(f"Enrichment rows upserted: {len(rows)}")
print(f"Enrichment failures: {failures}")


Distinct ISBN13s to enrich: 199
Enrichment rows upserted: 199
Enrichment failures: 31


In [6]:
# Validate final join for the selected date.
enriched = pd.read_sql_query(
    """
    SELECT
        n.list_name,
        n.published_date,
        n.rank,
        n.title,
        n.author,
        n.isbn13,
        e.work_key,
        e.subjects,
        e.subject_places,
        e.description,
        e.last_error,
        e.last_checked_at
    FROM nyt_entries n
    LEFT JOIN openlibrary_enrichment e
      ON e.isbn13 = n.isbn13
    WHERE n.published_date = ?
    ORDER BY n.list_name, n.rank
    """,
    conn,
    params=(published_date,),
)

enriched.head(30)


,list_name,published_date,rank,title,author,isbn13,work_key,subjects,subject_places,description,last_error,last_checked_at
0,"Advice, How-To & Miscellaneous",2024-06-16,1,ATOMIC HABITS,James Clear,9780735211292,/works/OL17930368W,Habit|Habit breaking|Behavior modification|Sel...,,"No matter your goals, Atomic Habits offers a p...",NaN,2026-02-23T01:43:22+00:00
1,"Advice, How-To & Miscellaneous",2024-06-16,2,GOOD ENERGY,Casey Means with Calley Means,9780593712641,/works/OL37826565W,nyt:advice-how-to-and-miscellaneous=2024-06-02...,,"What if depression, anxiety, infertility, inso...",NaN,2026-02-23T01:43:22+00:00
2,"Advice, How-To & Miscellaneous",2024-06-16,3,YOU DESERVE GOOD GELATO,Kacie Rose,9780593840436,/works/OL37566612W,nyt:advice-how-to-and-miscellaneous=2024-06-16...,,NaN,NaN,2026-02-23T01:43:22+00:00
3,"Advice, How-To & Miscellaneous",2024-06-16,4,THE NEW MENOPAUSE,Mary Claire Haver,9780593796252,/works/OL37588521W,nyt:advice-how-to-and-miscellaneous=2024-05-19...,,NaN,NaN,2026-02-23T01:43:22+00:00
4,"Advice, How-To & Miscellaneous",2024-06-16,5,MAKE YOUR BED,William H. McRaven,9781455570249,/works/OL17710052W,Inspirational|Self-Esteem|United States. Navy....,United States,Inspired by the advice Admiral William H. McRa...,NaN,2026-02-23T01:43:22+00:00
5,"Advice, How-To & Miscellaneous",2024-06-16,6,THE OFFICIAL STARDEW VALLEY COOKBOOK,ConcernedApe and Ryan Novak,9781984862051,/works/OL37627120W,Home economics|nyt:advice-how-to-and-miscellan...,,NaN,NaN,2026-02-23T01:43:22+00:00
6,"Advice, How-To & Miscellaneous",2024-06-16,7,THE CREATIVE ACT,Rick Rubin with Neil Strauss,9780593652886,/works/OL27955361W,New York Times Bestseller|creativity|Artists,,“I set out to write a book about what to do to...,NaN,2026-02-23T01:43:22+00:00
7,"Advice, How-To & Miscellaneous",2024-06-16,8,"THE BOY, THE MOLE, THE FOX AND THE HORSE",Charlie Mackesy,9780062976581,/works/OL21208765W,"Literature|Comics & graphic novels, literary|n...",,NaN,NaN,2026-02-23T01:43:22+00:00
8,"Advice, How-To & Miscellaneous",2024-06-16,9,"CHEAPER, FASTER, BETTER",Tom Steyer,9781954118645,/works/OL37629193W,nyt:advice-how-to-and-miscellaneous=2024-06-16...,,NaN,NaN,2026-02-23T01:43:22+00:00
9,"Advice, How-To & Miscellaneous",2024-06-16,10,MILLIONAIRE MISSION,Brian Preston,9781637745014,/works/OL37612895W,nyt:advice-how-to-and-miscellaneous=2024-06-16...,,NaN,NaN,2026-02-23T01:43:22+00:00


In [7]:
enriched.head(30)["subjects"][0]

'Habit|Habit breaking|Behavior modification|Self-actualization (psychology)|Business|psychology|Personal Growth|New York Times bestseller|BUSINESS & ECONOMICS / Organizational Behavior|PSYCHOLOGY / Social Psychology|SELF-HELP / Personal Growth / General.'

In [ ]:
# Optional quality checks
summary = pd.read_sql_query(
    """
    SELECT
      COUNT(*) AS rows_for_date,
      SUM(CASE WHEN e.isbn13 IS NOT NULL THEN 1 ELSE 0 END) AS matched_enrichment_rows,
      SUM(CASE WHEN e.description IS NOT NULL AND TRIM(e.description) <> '' THEN 1 ELSE 0 END) AS with_description,
      SUM(CASE WHEN e.subjects IS NOT NULL AND TRIM(e.subjects) <> '' THEN 1 ELSE 0 END) AS with_subjects,
      SUM(CASE WHEN e.subject_places IS NOT NULL AND TRIM(e.subject_places) <> '' THEN 1 ELSE 0 END) AS with_subject_places,
      SUM(CASE WHEN e.last_error IS NOT NULL AND TRIM(e.last_error) <> '' THEN 1 ELSE 0 END) AS with_errors
    FROM nyt_entries n
    LEFT JOIN openlibrary_enrichment e
      ON e.isbn13 = n.isbn13
    WHERE n.published_date = ?
    """,
    conn,
    params=(published_date,),
)
summary


In [5]:
conn.close()
